# Organized Cybersecurity XAI Notebook

Refactored and cleaned notebook structure for GitHub and academic use.

## Install Dependencies

In [ ]:
!pip install shap lime -q

## Imports

In [ ]:
# Importação de todas as bibliotecas que serão utilizadas
import pandas as pd
import numpy as np
import seaborn as sns
import shap
import lime
from lime import lime_tabular
import json
from openai import OpenAI
import matplotlib.pyplot as plt
"from sklearn.preprocessing import OneHotEncoderLabelEncoderStandardScaler
"from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
"from sklearn.metrics import classification_reportaccuracy_scoreconfusion_matrixConfusionMatrixDisplay
"from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize
"from sklearn.metrics import roc_curveaucprecision_recall_curve
"from sklearn.metrics import average_precision_score

## Dataset Loading

In [ ]:
"# Define o dataset que será utilizadodisponível em: https://www.kaggle.com/datasets/developerghost/intrusion-detection-logs-normal-bot-scan/code
"df = pd.read_csv("/content/sample_data/Network_logs.csv")

In [ ]:
# Cria uma cópia do dataset original para garantir que nenhum dado original foi alterado acidentalmente
networkData = df.copy()
networkData.head(5)

In [ ]:
# Descarta as features/colunas IP de Origem e IP de destino devido a alta cardinalidade e irrelevãncia e a feature Intrusion para evitar overfitting
"networkData.drop(['Source_IP''Destination_IP''Intrusion'

In [ ]:
# Codificação para as features/colunas com valores não numéricos
"categorical_cols = ['Request_Type''Protocol''User_Agent''Status''Port'

In [ ]:
for col in categorical_cols:
"    print(f"{col} categories: {networkData[col

In [ ]:
for col in categorical_cols:
"    networkData[col

In [ ]:
# Codificação da variável Alvo/Classificação (y) para valor numérico: BotAttack ->0; Normal ->1; ScanPort ->2;
target_encoder = LabelEncoder()
"networkData['Scan_Type_Label'

In [ ]:
"label_mapping = dict(zip(target_encoder.classes_target_encoder.transform(target_encoder.classes_)))
"print("Label Mapping:\label_mapping)"

In [ ]:
"networkData.drop(['Scan_Type'

In [ ]:
# Normalização/Transformação do valor númerico da feature/colunas Payload_size
scaler = StandardScaler()
"networkData['Payload_Size'

In [ ]:
networkData.head(5)

In [ ]:
# Define o que são as features para treinamento (X) e o que é o alvo/classificação (y)
"X = networkData.drop(['Scan_Type_Label'

## LLM Explainability

In [ ]:
# Informa a chave da API da OpenAI
client = OpenAI(api_key="")

In [ ]:
# Amostra para não enviar dataset inteiro para o LLM
"#sample_df = networkData.sample(n=min(50len(networkData))random_state=42)
"#sample_df

## Train/Test Split

In [ ]:
"# Particiona o dataset (networkData) em 70% para treinamento e 30% para testeuse o parâmetro "stratify" para assegurar uma proporção igual da classe "y" ("Scan_Type_Label") em cada conjunto (Treinamento e Teste)
""X_trainX_testy_trainy_test = train_test_split(
""    Xytest_size=0.3random_state=42stratify=y)"

## SMOTE Oversampling

In [ ]:
# Instancia o objeto de sobreamostragem SMOTE com as configurações padrão. Necessário para dataset desequilibrados
smote = SMOTE()

"# Aplica a técnica de sobreamostragem SMOTE aos dados de treinamento do modelo (NÃO aplicar aos dados de testepara evitar memorização) para equilibrar a classe minoritária da variável alvo (para equilibrar a quantidade de 0s e 1s).
""X_trainy_train = smote.fit_resample(X_trainy_train)
"
# Restaura y_train para uma série do pandas
"y_train = pd.Series(y_train.values.ravel()name='Scan_Type_Label')
"
# Mensagem de sucesso confirmando a nova distribuição da classe alvo/classificação
print('Técnica SMOTE aplicada com sucesso aos dados de treinamento.' + '\
' + '\
' + 'Nova distribuição de y alvo/classificação:' + '\
' + '\
' + str(y_train.value_counts()) + '\
' + '\
')

In [ ]:
"# Contagem de 0's1's e 2's da variável alvo/classificação.
"ScanTypeCounts = y_train.value_counts().sort_index()

# Cria uma figura com a nova distribuição equilibrada da variável alvo/classificação.
"figax = plt.subplots(figsize=(64))
"
# Cores para cada classe
"cores = ['#FF00FF''blue''#808000'

## Random Forest

In [ ]:
# Define a lista de algoritmos de classificação que serão utilizados para treinar o modelo
models = {
"    #'Logistic Regression': LogisticRegression(max_iter=1000)
""    #'KNN': KNeighborsClassifier()
""    #'Decision Tree': DecisionTreeClassifier()
""    'Random Forest': RandomForestClassifier()
""    #'XGBoost': xgb.XGBClassifier(use_label_encoder=Falseeval_metric='mlogloss')
"    #'MLP': MLPClassifier(max_iter=500)
}

## SHAP Explainability

In [ ]:
"# Seleciona uma amostra aleatória do conjunto de teste (X_test)limitada a no máximo 200 linhas para ser utilizado no SHAP
""sample_idx = np.random.choice(X_test.indexsize=min(200len(X_test))replace=False)
""X_sample = X_test.loc[sample_idx

## SHAP Explainability

In [ ]:
"for namemodel in models.items():
"    print(f"\
🔍 Treinamento do modelo utilizando o algorítmo {name}...")
"    model.fit(X_trainy_train)
"    y_pred = model.predict(X_test)

"    acc = accuracy_score(y_testy_pred)
"    print(f"✅ Acurácia: {acc:.4f}")
"    print(classification_report(y_testy_pred))
"
    print(f"📊 Explicação SHAP para {name}")

"    if name in ['Random Forest''XGBoost''Decision Tree'

In [ ]:
# View confusion matrix for test data and predictions
"confusion_matrix(y_testy_pred)"

## SHAP Explainability

In [ ]:
# Get and reshape confusion matrix data
"matrix = confusion_matrix(y_testy_pred)
""matrix = matrix.astype('float') / matrix.sum(axis=1)[:np.newaxis

In [ ]:
# =========================
    # 🔴 ROC CURVE
    # =========================
    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    for i in range(n_classes):
"        fpr[i

## SHAP Explainability

In [ ]:
feature_names = list(X.columns)

shap_global = {}
"for cls_idxcls_name in enumerate(class_names):
""    mean_abs = np.abs(shap_values[::cls_idx

In [ ]:
# ==========================
# DESCRIÇÃO DAS COLUNAS
# ==========================
column_description = {
    "Port": "Porta utilizada na comunicação\
"    "Request_Type": "Tipo de requisição\
""    "Protocol": "Protoloco da Camada de Transportesegundo o modelo OSIpodendo ser: TCP ou UDP"
"    "Payload_Size": "Tamanho do pacote (informação útil)\
"    "User_Agent": "Agente utilizado na comunicação\
""    "Status": "Status da requisçãopodendo ser: Success ou Failure"
""    "Scan_Type_Label": "Classificação da comunicaçãopodendo ser: normal ou botattack ou PortScan"
"}

# ==========================
# EQUIVALÊNCIA CATEGORIA -> CÓDIGO NUMÉRICO DAS COLUNAS
# ==========================
category_encoding = {

    "Request_Type": {
"        "DNS": 0
""        "FTP": 1
""        "HTTP": 2
""        "HTTPS": 3
""        "SMTP": 4
""        "SSH": 5
"        "Telnet": 6
"    }
"
    "Protocol": {
"        "ICMP": 0
""        "TCP": 1
"        "UDP": 2
"    }
"
    "User_Agent": {
"        "Mozilla/5.0": 0
""        "Nikto/2.1.6": 1
""        "Wget/1.20.3": 2
""        "curl/7.68.0": 3
""        "nmap/7.80": 4
"        "python-requests/2.25.1": 5
"    }
"
    "Status": {
"        "Failure": 0
"        "Success": 1
"    }
"
    "Port": {
"        21: 0
""        22: 1
""        23: 2
""        25: 3
""        53: 4
""        80: 5
""        135: 6
""        443: 7
""        4444: 8
""        6667: 9
""        8080: 10
"        31337: 11
"    }
"
    "Scan_Type_Label": {
"        "BotAttack": 0
""        "Normal": 1
"        "PortScan": 2
    }
}

# Estatísticas do Dataset
stats = networkData.describe(include="all").to_string()

# juntar X e y
#train_df = X_train.copy()
#train_df["Scan_Type_Label\

In [ ]:
train_sample_json

In [ ]:
pred_sample_json

## SHAP Explainability

In [ ]:
shap_global_json

In [ ]:
# ==========================
# PROMPT PARA O LLM
# ==========================
prompt = f"""
Você é especialista em Inteligência Artificial Explicável (XAI) e Segurança Cibernética.
Sua tarefa é analisar um modelo de aprendizado de máquina treinado e fornecer uma explicação objetiva e coesa de seu comportamento.

=========================
INFORMAÇÕES DO MODELO
=========================
{model_info}

=========================
DESCRIÇÃO DAS COLUNAS
=========================
{column_description}

=========================
CODIFICAÇÃO: CATEGORIA → VALOR NUMÉRICO
=========================
{category_encoding}

=========================
AMOSTRA DOS DADOS DE TREINAMENTO
=========================
{train_sample_json}

=========================
AMOSTRA DA CLASSIFICAÇÃO REAL vs CLASSIFICAÇÃO DO MODELO
=========================
{pred_sample_json}

=========================
TAREFA
=========================

"Forneça uma análise detalhada da explicabilidade do modeloincluindo:
"
1. Explique quais features são mais relevantes em cada classe (faça um ranking top-3).
2. Compare as diferenças entre as classes.
3. Interprete o comportamento do modelo no contexto de cibersegurança
4. Evite inferir causalidade — descreva apenas associações.


"Sua explicação deve ser técnicaporém claraadequada tanto para usuários leigos em cibersegurança e IAquanto para usuários com conhecimento avançado/especialistas.
""""

In [ ]:
# ==========================
# CHAMADA DA API
# ==========================
response = client.responses.create(
    model="gpt-5\
"    input=[
        {
            "role": "system\
"            "content": "Você é um especialista em Machine Learning e Explainable AI."
"        }
"        {
            "role": "user\
"            "content": prompt
        }

In [ ]:
# ==========================
# RESULTADO
# ==========================

explanation = ""

for item in response.output:
    if item.type == "message":
        for content in item.content:
            if content.type == "output_text":
                explanation += content.text


print("\
===== EXPLICAÇÃO DO MODELO =====\
")
print(explanation)